<a href="https://colab.research.google.com/github/Text-Machine/mask-predict/blob/main/sft_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

# Supervised Fine-Tuning Gemma on Prompt/Completion CSV

This notebook trains a Gemma instruction model using your CSV file with columns:
- `prompt`
- `completions`

It is configured for **Google Colab + A100 GPU** using **LoRA/QLoRA** for efficient fine-tuning.

In [ ]:
# 1) Install dependencies (Colab)
# Restart runtime after this cell if Colab had older preinstalled versions loaded.
!pip -q install -U "transformers>=4.43" "trl>=0.9.6" "peft>=0.11.1" "accelerate>=0.33.0" bitsandbytes datasets pandas scipy sentencepiece

In [ ]:
# 2) Imports and configuration
import os
import pandas as pd
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# ----------------------
# User settings
# ----------------------
CSV_PATH = "/content/your_data.csv"  # <-- upload your CSV to Colab and set path
MODEL_ID = "google/gemma-2-2b-it"    # alternatives: google/gemma-2-9b-it (larger)
OUTPUT_DIR = "/content/gemma-sft-lora"
MAX_SEQ_LENGTH = 1024
TEST_SIZE = 0.05
SEED = 42

# If the model is gated, make sure you are logged in on Hugging Face:
# from huggingface_hub import notebook_login
# notebook_login()

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
df = pd.read_csv('test_sft.csv')
df.columns
df['prompt'] = df.apply(lambda x: x.training_instruction + " " + x.prompt, axis=1)
df['completions'] = df['training_output']
df[['prompt','completions']].to_csv('test_sft_formatted.csv')

In [ ]:
# 3) Load CSV (must contain columns: prompt, completions)
df = pd.read_csv('test_sft_formatted.csv',index_col=0)
required_cols = {"prompt", "completions"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found: {list(df.columns)}")

df = df.dropna(subset=["prompt", "completions"]).copy()
df["prompt"] = df["prompt"].astype(str)
df["completions"] = df["completions"].astype(str)

print("Rows after cleaning:", len(df))
df.head()

In [ ]:
# 4) Build training/eval datasets with Gemma chat formatting
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

START_USER = "<start_of_turn>user\n"
START_MODEL = "<start_of_turn>model\n"
END_TURN = "<end_of_turn>\n"

def to_text(example):
    prompt = example["prompt"].strip()
    completion = example["completions"].strip()
    text = f"{START_USER}{prompt}{END_TURN}{START_MODEL}{completion}{END_TURN}"
    return {"text": text}

dataset = Dataset.from_pandas(df[["prompt", "completions"]], preserve_index=False)
dataset = dataset.map(to_text, remove_columns=dataset.column_names)
dataset = dataset.train_test_split(test_size=TEST_SIZE, seed=SEED)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(train_dataset)
print(eval_dataset)
print("Sample formatted record:\n", train_dataset[0]["text"][:500])

In [ ]:
# 5) Load Gemma in 4-bit and run supervised fine-tuning (LoRA/QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager",
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
    seed=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    args=sft_config,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Training complete. Adapter saved to:", OUTPUT_DIR)

In [ ]:
# 6) Quick inference test with the fine-tuned model
model.eval()

test_prompt = "Write a short answer about why public transport matters."
formatted = f"{START_USER}{test_prompt}{END_TURN}{START_MODEL}"
inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output[0], skip_special_tokens=False))

# Optional: zip adapter for download from Colab
# !zip -r /content/gemma-sft-lora.zip /content/gemma-sft-lora

## 7) Push Fine-Tuned Adapter to Hugging Face

This uploads the LoRA adapter saved in `OUTPUT_DIR` to your Hugging Face model repository.

In [ ]:
# 8) Login and set your Hugging Face repo details
from huggingface_hub import notebook_login, HfApi

# Run and paste your HF token when prompted
notebook_login()

HF_REPO_ID = "Kaspar/gemma-2-2b-it-sft-lora"  # <-- change this
PRIVATE_REPO = False

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=PRIVATE_REPO, exist_ok=True)
print("Repo ready:", HF_REPO_ID)

In [ ]:
# 9) Upload local adapter folder to Hugging Face Hub
# Make sure training cell has run and OUTPUT_DIR exists.
api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type="model",
    folder_path=OUTPUT_DIR,
    commit_message="Upload Gemma SFT LoRA adapter",
)

print(f"Uploaded adapter from {OUTPUT_DIR} to https://huggingface.co/{HF_REPO_ID}")

## 10) Compare Original vs Fine-Tuned Model (Loaded From Hugging Face Hub)

The next cells load both models from Hugging Face Hub:
- original base model from `MODEL_ID`
- fine-tuned adapter from `HF_REPO_ID`

Then they generate side-by-side outputs for the same prompts.

In [ ]:
# 11) Load base model and fine-tuned model (adapter) from Hugging Face Hub
from peft import PeftModel
from huggingface_hub import HfApi

HF_REPO_ID = "Kaspar/gemma-2-2b-it-sft-lora"
MODEL_ID = "google/gemma-2-2b-it"

compare_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if compare_tokenizer.pad_token is None:
    compare_tokenizer.pad_token = compare_tokenizer.eos_token

compare_bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Adapter must already be uploaded to this repo in Section 9.
FT_ADAPTER_REPO = HF_REPO_ID

# Optional sanity check that the model repo exists and is reachable.
api = HfApi()
_ = api.model_info(FT_ADAPTER_REPO)

print("Loading base model from hub:", MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=compare_bnb,
    device_map="auto",
    attn_implementation="eager",
)
base_model.eval()

print("Loading fine-tuned model from hub (base + adapter):", FT_ADAPTER_REPO)
ft_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=compare_bnb,
    device_map="auto",
    attn_implementation="eager",
)
fine_tuned_model = PeftModel.from_pretrained(ft_base_model, FT_ADAPTER_REPO)
fine_tuned_model.eval()

print("Both models loaded from Hugging Face Hub.")

In [ ]:
# 12) Generate side-by-side outputs for the same prompts
import pandas as pd

compare_prompts = [
    "Write a concise explanation of why renewable energy matters.",
    "Draft a polite email asking for a meeting next week.",
    "Explain gradient descent in simple terms for a beginner.",
]

# Optional: pull a few prompts from your dataset for comparison
if "df" in globals() and "prompt" in df.columns:
    dataset_prompts = df["prompt"].dropna().astype(str).head(2).tolist()
    compare_prompts.extend(dataset_prompts)

def generate_response(model_obj, prompt, max_new_tokens=160):
    formatted = f"{START_USER}{prompt}{END_TURN}{START_MODEL}"
    inputs = compare_tokenizer(formatted, return_tensors="pt").to(model_obj.device)
    with torch.no_grad():
        outputs = model_obj.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            top_p=1.0,
            eos_token_id=compare_tokenizer.eos_token_id,
            pad_token_id=compare_tokenizer.eos_token_id,
        )
    decoded = compare_tokenizer.decode(outputs[0], skip_special_tokens=False)

    # Keep only model answer region for cleaner comparison
    split_key = START_MODEL
    answer = decoded.split(split_key, 1)[-1]
    answer = answer.split("<end_of_turn>", 1)[0].strip()
    return answer

rows = []
for p in compare_prompts:
    base_out = generate_response(base_model, p)
    ft_out = generate_response(fine_tuned_model, p)
    rows.append({
        "prompt": p,
        "base_output": base_out,
        "fine_tuned_output": ft_out,
    })

comparison_df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 400)
comparison_df